# Integrating a reinforcement-learning agent with Giulia

This notebook describes the general pattern for exposing Giulia as an environment for a
sequential decision process. It is the reference for Section 4.2 of the report: the classes
here contain no scenario-specific quantity, and a study is expected to reuse them by
supplying the four definitions of the next section.

A concrete realization is given at the end as an example, not as the interface itself.


## The four definitions an integration supplies

Chapter 2 describes the two interfaces through which an external component interacts with
Giulia: the **read path**, in which each module declares the quantities it exposes through
`SaveAble`, and the **write path**, in which Dedalo applies configuration changes to the
engineering-parameter tables.

Any over-the-top integration is fully specified by four definitions expressed against those
two interfaces:

1. **Observed quantities** — which exposed keys are read, and how they are aggregated.
2. **Modifiable parameters** — which engineering-parameter columns are written, and the
   admissible values of each.
3. **Point of application** — the stage of the simulation lifecycle at which a change takes
   effect.
4. **Objective** — how the measured quantities are converted into a reward or a score.

Everything outside these four definitions — the learning algorithm, the surrogate model, the
acquisition rule — is external to the simulator and can be replaced without modifying them.
The classes below isolate the four definitions as the only methods a study needs to supply.


## The write path

Dedalo is the interface through which the engineering-parameter tables are modified. The
subclass below is the smallest one an integration can declare.

In [ ]:
"""A minimal Dedalo subclass.

Dedalo declares two abstract stages. A study that changes nothing per episode or per step
still has to define them, because the lifecycle is a requirement of the interface rather
than an implementation convention. Study-specific configuration goes here.
"""

from dedalo.dedalo import Dedalo


class OttDedalo(Dedalo):

    def process_static(self, episode_index: int = 0, **kwargs) -> None:
        """Applied once per episode, before `configure()`.

        Cell-side settings belong here: the UE tables do not exist yet.
        """
        return None

    def process_realTime(self, step_index: int = 0, **kwargs) -> None:
        """Applied once per step, before `run_simulation()`.

        UE-side settings belong here: the UE tables are rebuilt by `configure()`.
        """
        return None

## Point of application

The ordering below is the same for every integration and follows the Dedalo stages of
Section 2.8. It is written once, in the base class, so that a study cannot place the calls
inconsistently between its initialization and its subsequent steps:

| stage | call | what it may change |
|---|---|---|
| once per episode | `process_static(episode_index)` | cell-side parameters |
| once per episode | `configure(episode_index, max_simulation_time_us)` | builds the UE and link side |
| once per interval | `process_realTime(step_index)` | UE-side parameters |
| once per interval | `run_simulation(step_index, decision_interval_us)` | advances the event-driven engine |

An action is written before the interval it is meant to affect, and a quantity read through
`SaveAble` after `run_simulation()` belongs to the interval that call has just completed.
Reading before it returns the previous interval.

One decision is not one simulation interval. `reset()` consumes one interval to establish
the initial observation, before any action has been taken, so an episode of *n* decisions
advances the engine *n+1* times. Reporting the two counts separately is what allows a
training budget to be compared with a simulation cost.


## The environment

The interaction sequence is written once. The four definitions are the only methods a study
overrides.

In [ ]:
"""A generic reinforcement-learning environment over Giulia.

The interaction sequence is fixed. A study supplies the four definitions by overriding
`apply_action`, `build_observation`, `reward` and, optionally, `is_terminated`.

Two counters are kept apart, because they are not the same quantity. `_step_index` is the
simulation interval passed to the engine and advances monotonically, including for the
interval that establishes the initial observation. `_decisions` counts the actions the
policy has taken, and is what the episode horizon is expressed in.
"""

from typing import Any, Dict, List, Tuple

from giulia.outputs.saveable import Saveable


class GiuliaEnvironment:

    def __init__(self, giulia, dedalo, *,
                 episode_index: int = 0,
                 decision_interval_us: float = 1e6,
                 steps_per_episode: int = 10,
                 max_simulation_time_us: float | None = None,
                 observed_keys: List[str] | None = None):
        self.giulia = giulia
        self.dedalo = dedalo
        self.episode_index = episode_index
        self.decision_interval_us = decision_interval_us
        self.steps_per_episode = steps_per_episode
        # The engine stops executing events once the clock reaches the horizon given
        # to `configure()`. An episode needs one interval per decision plus the one
        # that establishes the initial observation, so the default is derived rather
        # than guessed: a horizon that is too short silently produces empty steps.
        self.max_simulation_time_us = (
            max_simulation_time_us if max_simulation_time_us is not None
            else decision_interval_us * (steps_per_episode + 1))
        self.observed_keys = observed_keys or []
        self._step_index = 0
        self._decisions = 0

    # ---- the four definitions ------------------------------------------
    def apply_action(self, action: Any) -> None:
        """Write the action to the engineering-parameter tables through Dedalo."""
        raise NotImplementedError

    def build_observation(self) -> Dict[str, Any]:
        """Read the exposed quantities and aggregate them into an observation."""
        return {key: Saveable.get(key) for key in self.observed_keys}

    def reward(self) -> float:
        """Convert the measured quantities into a scalar reward."""
        raise NotImplementedError

    def is_terminated(self) -> bool:
        return self._decisions >= self.steps_per_episode

    # ---- the fixed interaction sequence --------------------------------
    def _advance(self) -> None:
        """One simulation interval: configure the step, then run it."""
        self.dedalo.process_realTime(step_index=self._step_index)
        self.giulia.run_simulation(self._step_index, self.decision_interval_us)
        self._step_index += 1

    def reset(self) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        self.dedalo.process_static(episode_index=self.episode_index)
        self.giulia.configure(self.episode_index, self.max_simulation_time_us)
        self._step_index = 0
        self._decisions = 0
        self._advance()                      # establishes the initial observation
        return self.build_observation(), {}

    def step(self, action: Any):
        self.apply_action(action)            # written before the interval it affects
        self._advance()
        self._decisions += 1
        observation = self.build_observation()
        return observation, self.reward(), self.is_terminated(), False, {}

## A deliberately trivial instance

The clearest way to see that the four definitions are all a study supplies is to write one
whose network problem carries no interest of its own.

A single \ac{UE} is moved one step at a time on the playground, and it is rewarded for
*reducing* its own throughput. The problem is meaningless as radio engineering, which is the
point: everything that remains in the cell below is integration machinery, and each of the
four definitions occupies one method.

The same example, written against the raw simulator attributes, is the starting point of the
`RL_Giulia_Template` repository. The version below differs in one respect: the position is
written through the Dedalo setter rather than assigned into `ue_deployment_obj.df_ep`, and
the throughput is read through `SaveAble.get()` rather than from `performance_obj`. Nothing
else changes, and the comparison is a compact illustration of what the two interfaces buy.


In [ ]:
"""An instance with no engineering content: one UE moving on the playground.

Four definitions, one method each:

  apply_action      -> which parameters change   (UE position, through Dedalo)
  build_observation -> what is observed          (position and throughput, through SaveAble)
  reward            -> what "good" means         (a lower throughput for that UE)
  is_terminated     -> when the episode ends     (inherited)
"""

import numpy as np
from giulia.outputs.saveable import Saveable


class SingleUeDedalo(OttDedalo):
    """The controlled UE is positioned by the agent, so its mobility model is disabled.

    This belongs to `process_realTime` rather than to `process_static`: the UE table is
    rebuilt by `configure()` and does not exist before it.
    """

    def __init__(self, giulia, ue_name: str = "UE_0"):
        super().__init__(giulia)
        self.ue_name = ue_name

    def process_realTime(self, step_index: int = 0, **kwargs) -> None:
        if step_index == 0:
            self.set_giulia_ue_df_ep(
                name_ue_list=[self.ue_name],
                name_attribute="mobility_model",
                value_list=[None])


class SingleUeMobilityEnv(GiuliaEnvironment):

    THROUGHPUT_KEY = "Rate_based_on_eff_CSI_RS_SINR.ue_throughput_Mbps"

    #: the four moves, in metres
    MOVES = {0: (+10.0, 0.0), 1: (0.0, +10.0), 2: (-10.0, 0.0), 3: (0.0, -10.0)}

    def __init__(self, giulia, dedalo, ue_name: str = "UE_0",
                 bounds_m: tuple[float, float] = (-1500.0, 1500.0), **kwargs):
        super().__init__(giulia, dedalo,
                         observed_keys=[self.THROUGHPUT_KEY], **kwargs)
        self.ue_name = ue_name
        self.low_m, self.high_m = bounds_m

    # ---- helpers reading the UE table ----------------------------------
    def _position(self) -> np.ndarray:
        table = self.giulia.ue_deployment_obj.df_ep.set_index("name")
        return np.array([table.loc[self.ue_name, "position_x_m"],
                         table.loc[self.ue_name, "position_y_m"]], dtype=float)

    def _ue_index(self) -> int:
        names = self.giulia.ue_deployment_obj.df_ep["name"].tolist()
        return names.index(self.ue_name)

    # 2. modifiable parameters, written through the Dedalo interface
    def apply_action(self, action) -> None:
        dx, dy = self.MOVES[int(action)]
        x, y = np.clip(self._position() + (dx, dy), self.low_m, self.high_m)
        for attribute, value in (("position_x_m", x), ("position_y_m", y)):
            self.dedalo.set_giulia_ue_df_ep(
                name_ue_list=[self.ue_name],
                name_attribute=attribute,
                value_list=[float(value)])

    # 1. observed quantities, read through the SaveAble interface
    def build_observation(self):
        throughput = np.asarray(Saveable.get(self.THROUGHPUT_KEY)).squeeze()
        x, y = self._position()
        return {"throughput_Mbps": float(throughput[self._ue_index()]),
                "position_x_m": x,
                "position_y_m": y}

    # 4. objective
    def reward(self) -> float:
        return -self.build_observation()["throughput_Mbps"]

## A second instance

The realistic case used in the report controls the activity of the terrestrial capacity cells
instead of a position. The interaction sequence, the execution order and the read path are
unchanged; only the same four methods are rewritten.

In [ ]:
"""An instance: terrestrial carrier activity.

Only the four definitions are supplied. The interaction sequence, the lifecycle ordering
and the read path are inherited unchanged.
"""

import numpy as np
from giulia.outputs.saveable import Saveable


class CarrierActivityEnv(GiuliaEnvironment):

    #: engineering-parameter column written by the action
    ACTION_COLUMN = "radio_unit_initial_status"

    def __init__(self, giulia, dedalo, controlled_cells: list[str], **kwargs):
        super().__init__(
            giulia, dedalo,
            observed_keys=["BaseStations.prb_usage_per_cell",
                           "BaseStations.cell_radio_unit_statuses"],
            **kwargs)
        self.controlled_cells = controlled_cells

    # 2. modifiable parameters, written through the Dedalo interface
    def apply_action(self, action) -> None:
        states = ["active" if bit else "carrier_shutdown" for bit in np.asarray(action).ravel()]
        self.dedalo.set_giulia_network_df_ep(
            name_cell_list=self.controlled_cells,
            name_attribute=self.ACTION_COLUMN,
            value_list=states)

    # 1. observed quantities, read through the SaveAble interface
    def build_observation(self):
        raw = super().build_observation()
        usage = np.asarray(raw["BaseStations.prb_usage_per_cell"], dtype=float)
        status = np.asarray(raw["BaseStations.cell_radio_unit_statuses"], dtype=str)
        return {"prb_usage": usage,
                "active": (status == "active").astype(np.float32)}

    # 4. objective
    def reward(self) -> float:
        observation = self.build_observation()
        return float(-observation["active"].mean())

## The learning library is external

In [ ]:
"""Connecting a learning library.

The environment above already exposes the `reset()` / `step()` signatures that Gymnasium
expects. Wrapping it for a specific library is therefore an adapter, not a redesign, and it
does not belong to the four definitions.

Neither Gymnasium nor Stable-Baselines3 is installed in the project environment, so the
adapter below is guarded. The integration itself does not depend on either package.
"""

try:
    import gymnasium as gym
    from gymnasium import spaces

    class GymAdapter(gym.Env):
        def __init__(self, environment: GiuliaEnvironment, n_controlled: int):
            self.environment = environment
            self.action_space = spaces.MultiBinary(n_controlled)
            self.observation_space = spaces.Dict({
                "prb_usage": spaces.Box(0.0, 1.0, shape=(n_controlled,)),
                "active":    spaces.Box(0.0, 1.0, shape=(n_controlled,)),
            })

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            return self.environment.reset()

        def step(self, action):
            return self.environment.step(action)

    print("Gymnasium adapter available.")

except ImportError:
    print("Gymnasium is not installed in this environment; "
          "the adapter is skipped and the integration above is unaffected.")

## Executing the loop

In [ ]:
"""Running it.

The cell below builds a real Giulia instance and executes a few decisions of the trivial
example. It is the only expensive cell in this notebook: a scenario build takes on the order
of a minute, and each decision advances the event-driven engine over one decision interval.
"""

RUN = False   # set to True to execute

if RUN:
    import numpy as np
    from giulia.presets.g_standard_preset import Giulia

    g = Giulia("GiuliaStd", "3GPPTR38_901_4G_5G_multilayer", None,
               "uniform", None, "downlink", False, False, False, 1, False, False)
    dedalo = SingleUeDedalo(g, ue_name="UE_0")

    env = SingleUeMobilityEnv(g, dedalo, ue_name="UE_0", steps_per_episode=5)

    observation, _ = env.reset()
    print("initial:", observation)

    rng = np.random.default_rng(0)
    terminated = False
    while not terminated:
        action = int(rng.integers(0, 4))
        observation, reward, terminated, _, _ = env.step(action)
        print(f"action={action}  reward={reward:+.3f}  "
              f"position=({observation['position_x_m']:.0f}, "
              f"{observation['position_y_m']:.0f})")

    print(f"decisions={env._decisions}  simulation intervals={env._step_index}")

## What a study changes

| definition | where it is supplied | trivial instance | carrier instance |
|---|---|---|---|
| observed quantities | `observed_keys`, `build_observation` | one UE's throughput and position | per-cell PRB usage and radio status |
| modifiable parameters | `apply_action` | `position_x_m`, `position_y_m` | `radio_unit_initial_status` |
| point of application | inherited | `process_realTime` before `run_simulation` | the same |
| objective | `reward` | the negated throughput | mean active fraction |

Neither instance changes the interaction sequence, and neither is aware of the learning
algorithm. Replacing the observed keys, the written columns or the objective produces a
different study; replacing the algorithm produces none of the three.